# Single-Agent Human-in-the-Loop: Gating a Dynamically-Chosen Tool Call

## Problem card

- **User/trigger:** the same on-call SRE investigation from
  `01_react_single_agent.ipynb`, extended with one new, real,
  **mutating** capability: `restart_service(service_name)`.
- **Inputs:** identical to notebook 01 -- logs, deploys, metrics,
  dependency health -- plus the new remediation tool.
- **Output:** an incident report, and, only if the agent decides
  remediation is warranted *and* a human approves it, an actual service
  restart.
- **Success criteria:** the agent can still investigate freely (any tool,
  any order, exactly like notebook 01), but the moment it decides to call
  the mutating tool -- whenever that happens in its own reasoning
  trajectory -- a human is asked to approve it first.
- **Topology:** still a **true bounded ReAct agent**, unchanged from
  notebook 01. This notebook does not add a planner, a supervisor, or a
  fixed graph position for the risky step -- it adds a **gate that
  travels with the tool**, not with a place in the graph.

## Why this notebook exists, and why it isn't just `agent_hitl.ipynb` again

`agent_hitl.ipynb` (in the parent `langgraph_basics/` folder) gates
**fixed graphs** -- small, purpose-built pipelines where the risky action
sits at one specific, known node (`approval`, `act`). That works because
the graph's shape *is* the plan.

A true ReAct agent has no such shape. `investigation_agent` from notebook
01 has exactly one tool-execution step (`ToolNode`), reused every single
time the agent calls anything -- `get_service_logs` on iteration 1,
`get_recent_deploys` on iteration 3, maybe `get_service_logs` again on
iteration 4. There is no "the node right before the risky action" to
attach a static `interrupt_before` to, because the risky action can be
requested at **any** iteration, or not at all, depending on what the
agent finds. Gating it requires intercepting tool calls **by name**,
inside the one shared execution step, regardless of when that name comes
up -- a different mechanic from anything in `agent_hitl.ipynb`, and the
reason this is its own notebook rather than a shared example.

## Series position

This is a direct extension of `01_react_single_agent.ipynb` -- same
fixtures, same agent, same checkpointing conventions as that notebook and
`agent_memory_deepdive.ipynb`. It is not part of that folder's numbered
five-notebook progression (00-04); it's a focused companion piece,
answering the question notebook 01's own "Other design considerations"
section raised and deliberately left open: *"a rollback or
configuration-change tool would need approval, idempotency, and an
explicit human-review boundary"* -- this notebook is that boundary,
built and run for real.

## Setup

In [1]:
import os
import time as _time
import warnings
import logging
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

load_dotenv(find_dotenv(usecwd=True))
PROVIDER = os.getenv("PROVIDER", "openai").lower()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

import shared  # scorecard helpers (standardized single-agent scorecard)

scorecard = shared.ScorecardCallback()


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=key, callbacks=[scorecard])
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.3, api_key=key, callbacks=[scorecard])
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o-mini


## The fixtures and tools: notebook 01's investigation tools, plus one mutating action

Reused verbatim from `01_react_single_agent.ipynb`: the same incident
data shape, the same four read-only investigation tools. The only
addition is `restart_service` -- a real, irreversible action (it resets
the service's error-rate fixture, simulating an actual restart's effect)
and a log of what was actually executed, so approvals vs. rejections are
independently verifiable, not just inferred from printed text.

In [2]:
from langchain_core.tools import tool

_LOGS = {
    "checkout-service": (
        "14:32:01 ERROR checkout-service: connection pool exhausted, "
        "waiting on payments-db connection (timeout after 5000ms)\n"
        "14:32:03 ERROR checkout-service: connection pool exhausted, "
        "waiting on payments-db connection (timeout after 5000ms)\n"
        "14:31:58 WARN checkout-service: connection pool at 100% utilization "
        "(configured max: 5, previous config max: 50)"
    ),
    "payments-db": (
        "14:32:00 INFO payments-db: healthy, 340 active connections, "
        "12ms avg query latency, no errors in last 30 minutes"
    ),
    "inventory-service": (
        "09:14:02 ERROR inventory-service: upstream warehouse-api returned 503 "
        "for 40% of requests over the last 5 minutes\n"
        "09:14:05 ERROR inventory-service: circuit breaker OPEN for warehouse-api"
    ),
}

_DEPLOYS = {
    "checkout-service": [
        {"id": "PAY-2231", "deployed_at": "14:12:00", "summary": "Refactor DB connection pooling config"},
        {"id": "PAY-2198", "deployed_at": "09:03:00", "summary": "Add loyalty-points display to cart"},
    ],
    "inventory-service": [],  # no recent deploys -- rules out that path deliberately
}

_METRICS = {
    ("checkout-service", "error_rate"): "Error rate: 0.2% baseline -> 34% starting 14:32, still elevated",
    ("checkout-service", "latency_p99"): "p99 latency: 220ms baseline -> 5200ms starting 14:32",
    ("payments-db", "error_rate"): "Error rate: 0.01%, no change in last 2 hours",
    ("inventory-service", "error_rate"): "Error rate: 0.1% baseline -> 41% starting 09:14",
}

_DEPENDENCIES = {
    "checkout-service": ["payments-db", "inventory-service", "auth-service"],
    "inventory-service": ["warehouse-api"],
}

_RESTARTS_EXECUTED = []


@tool
def get_service_logs(service_name: str) -> str:
    "Fetch recent error/warning logs for a given service."
    return _LOGS.get(service_name, f"No log entries found for '{service_name}' in the last hour.")


@tool
def get_recent_deploys(service_name: str) -> str:
    "List deployments to a service in the last 24 hours, most recent first."
    deploys = _DEPLOYS.get(service_name, [])
    if not deploys:
        return f"No deploys found for '{service_name}' in the last 24 hours."
    return "\n".join(f"{d['id']} at {d['deployed_at']}: {d['summary']}" for d in deploys)


@tool
def get_service_metrics(service_name: str, metric: str) -> str:
    "Fetch a named metric (error_rate or latency_p99) for a service."
    return _METRICS.get((service_name, metric), f"No data for metric '{metric}' on '{service_name}'.")


@tool
def get_dependency_health(service_name: str) -> str:
    "List the services a given service depends on."
    deps = _DEPENDENCIES.get(service_name, [])
    return f"{service_name} depends on: {', '.join(deps)}" if deps else f"No dependency data for '{service_name}'."


@tool
def restart_service(service_name: str) -> str:
    "Restart a service. This is a REAL, irreversible remediation action, not a diagnostic."
    _RESTARTS_EXECUTED.append(service_name)
    _METRICS[(service_name, "error_rate")] = f"Error rate: reset to 0.1% baseline immediately after restart"
    return f"{service_name} restarted successfully. Error rate reset to baseline."


READ_ONLY_TOOLS = [get_service_logs, get_recent_deploys, get_service_metrics, get_dependency_health]
ALL_TOOLS = READ_ONLY_TOOLS + [restart_service]
ALL_TOOLS_BY_NAME = {t.name: t for t in ALL_TOOLS}
SENSITIVE_TOOLS = {"restart_service"}
print(f"{len(ALL_TOOLS)} tools registered, {len(SENSITIVE_TOOLS)} marked sensitive: {SENSITIVE_TOOLS}")


5 tools registered, 1 marked sensitive: {'restart_service'}


## The interceptor: a custom tool-execution node, not `ToolNode`

Notebook 01 used LangGraph's prebuilt `ToolNode` -- it executes whatever
tool calls the agent's last message requested, opaquely, with no hook to
intervene per-call. That's exactly right for an all-read-only tool set,
and exactly wrong once one tool can change something real: `ToolNode` has
no concept of "pause before this specific one." Gating requires writing
that tool-execution step by hand.

### Design

`tools_node_with_gate` replaces `ToolNode` in the graph. For every tool
call in the agent's latest message (there can be more than one per
turn), it checks the call's **name** -- not its position, not which
iteration this is -- against `SENSITIVE_TOOLS`. A match calls
`interrupt()` with the proposed call surfaced in full; everything else
executes immediately, exactly as `ToolNode` would have. The graph edges
around it (`agent -> tools -> agent`, `tools_condition` for finalize) are
**unchanged** from notebook 01 -- the gate is entirely inside one node,
not a new graph shape.

In [3]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from pydantic import BaseModel, Field


class IncidentReport(BaseModel):
    root_cause: str = Field(description="The specific, concrete root cause identified")
    affected_service: str
    contributing_deploy: str = Field(description="Deploy ID if a deploy caused this, else 'none'")
    confidence: Literal["low", "medium", "high"]
    recommended_action: str


class InvestigationState(TypedDict):
    messages: Annotated[list, add_messages]
    report: dict | None


llm_with_tools = llm.bind_tools(ALL_TOOLS)


def agent_node(state: InvestigationState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def tools_node_with_gate(state: InvestigationState) -> dict:
    last_message = state["messages"][-1]
    results = []
    for call in last_message.tool_calls:
        if call["name"] in SENSITIVE_TOOLS:
            decision = interrupt({
                "action": call["name"],
                "args": call["args"],
                "question": f"Approve {call['name']}({call['args']}) before it executes? This is a real, irreversible action.",
            })
            if not decision:
                results.append(ToolMessage(content=f"{call['name']} REJECTED by human reviewer -- not executed.", tool_call_id=call["id"]))
                continue
        tool_fn = ALL_TOOLS_BY_NAME[call["name"]]
        result = tool_fn.invoke(call["args"])
        results.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return {"messages": results}


def finalize_node(state: InvestigationState) -> dict:
    structured = llm.with_structured_output(IncidentReport)
    report = structured.invoke(
        state["messages"] + [HumanMessage(content="Based on the investigation above, produce the final incident report.")]
    )
    return {"report": report.model_dump()}


investigation_builder = StateGraph(InvestigationState)
investigation_builder.add_node("agent", agent_node)
investigation_builder.add_node("tools", tools_node_with_gate)
investigation_builder.add_node("finalize", finalize_node)
investigation_builder.add_edge(START, "agent")
investigation_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": "finalize"})
investigation_builder.add_edge("tools", "agent")
investigation_builder.add_edge("finalize", END)

STEP_BUDGET = 12
gated_investigation_agent = investigation_builder.compile(checkpointer=InMemorySaver())
print("Gated ReAct investigation agent compiled -- same shape as notebook 01, plus one interceptor inside 'tools'.")


Gated ReAct investigation agent compiled -- same shape as notebook 01, plus one interceptor inside 'tools'.


### Mermaid -- same topology as notebook 01, the gate is invisible at this level

```mermaid
graph TD
    START([START]) --> agent[agent: chooses next tool, any order]
    agent -->|tool call requested| tools[tools: executes each call --<br/>SENSITIVE ones interrupt() first]
    tools --> agent
    agent -->|no more tool calls| finalize[finalize: structured IncidentReport]
    finalize --> END([END])
```

This is the point worth sitting with: **the graph diagram did not change
from notebook 01.** The gate is not a new node position: `tools` is still
the one shared step every tool call passes through, on any iteration.
What changed is entirely inside that one node's logic.

## Real run 1: an incident where remediation is actually warranted

`checkout-service` -- the same root cause as notebook 01 (connection
pool misconfigured by deploy `PAY-2231`). This time the agent has
`restart_service` available and may reasonably decide, after identifying
the cause, that restarting clears the immediate symptom while the
misconfigured deploy gets properly rolled back separately.

In [4]:
SYSTEM_PROMPT = (
    "You are an SRE investigation agent. Use the available tools to find the root "
    "cause of the reported incident before answering. Check whatever systems the "
    "evidence points to -- don't follow a fixed checklist. If you identify a clear "
    "root cause and believe restarting the affected service would help mitigate "
    "customer impact while a proper fix is prepared, you may call restart_service."
)

# Scorecard capture starts here -- this is the notebook's representative measured
# run (both real HITL scenarios below), reset immediately before it.
scorecard.reset()
_scorecard_start = _time.time()
_human_interventions = 0

config_1 = {"configurable": {"thread_id": "hitl-checkout-incident"}, "recursion_limit": STEP_BUDGET}
result_1 = gated_investigation_agent.invoke({
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content="checkout-service error rate spiked around 14:32. Investigate and remediate if appropriate."),
    ],
    "report": None,
}, config=config_1)

print("=== TOOL-CALL TRACE ===")
step_index = 0
sensitive_fired_at = None
for m in result_1["messages"]:
    if hasattr(m, "tool_calls") and m.tool_calls:
        for tc in m.tool_calls:
            step_index += 1
            marker = " <-- SENSITIVE" if tc["name"] in SENSITIVE_TOOLS else ""
            print(f"  [{step_index}] called {tc['name']}({tc['args']}){marker}")
            if tc["name"] in SENSITIVE_TOOLS and sensitive_fired_at is None:
                sensitive_fired_at = step_index

if "__interrupt__" in result_1:
    print("\nPAUSED at tool-call index", sensitive_fired_at, "-- payload:", result_1["__interrupt__"])
else:
    print("\nNo sensitive tool call requested on this run.")


=== TOOL-CALL TRACE ===
  [1] called get_service_metrics({'service_name': 'checkout-service', 'metric': 'error_rate'})
  [2] called get_service_logs({'service_name': 'checkout-service'})
  [3] called get_recent_deploys({'service_name': 'checkout-service'})
  [4] called get_dependency_health({'service_name': 'checkout-service'})
  [5] called get_dependency_health({'service_name': 'payments-db'})

No sensitive tool call requested on this run.


In [5]:
# A human reviews the proposed restart and approves it.
if "__interrupt__" in result_1:
    _human_interventions += 1
    result_1 = gated_investigation_agent.invoke(Command(resume=True), config=config_1)
    print("Resumed after approval.")

print("\n=== FINAL REPORT ===")
print(result_1["report"])
print("\nRestarts actually executed:", _RESTARTS_EXECUTED)



=== FINAL REPORT ===
{'root_cause': 'The spike in error rates in the checkout-service was caused by a recent deployment that altered the database connection pooling configuration, reducing the maximum number of connections available to payments-db, leading to an exhausted connection pool.', 'affected_service': 'checkout-service', 'contributing_deploy': 'PAY-2231', 'confidence': 'high', 'recommended_action': 'Restart the checkout-service to clear the current state and allow it to re-establish connections to payments-db.'}

Restarts actually executed: []


**Expected output, and what actually happened on this run**: read the
actual trace above rather than assuming a fixed outcome. On this run, the
agent investigated `checkout-service`, correctly identified the
`PAY-2231` connection-pool misconfiguration, and its `recommended_action`
text says to restart the service -- but it **never actually called
`restart_service`**. `_RESTARTS_EXECUTED` is empty after this run, and no
`__interrupt__` ever fired, so there was nothing for a human to approve.

This is a real, honest divergence from an earlier run of this same
notebook (which did call `restart_service` and paused for approval), and
it's worth reading precisely rather than as a bug: the agent's free-text
`recommended_action` recommending a restart and its actual tool-call
behavior *not* requesting one are two different things, decided by two
different parts of the same model call (the ReAct loop's tool-call
decision vs. `finalize_node`'s separate structured-report generation) --
and on this run they disagreed with each other. This is itself a useful,
generalizable lesson the gate's existence doesn't fully solve: a gate can
only review a tool call that is actually made. A report that *recommends*
an action without the agent *requesting* it slips past any interceptor
entirely, silently, because there was never a call to intercept.

## Real run 2: an incident where restarting would NOT fix anything

`inventory-service` -- root cause is a downstream dependency
(`warehouse-api`) failing, not the service itself; `_DEPLOYS["inventory-service"]`
is deliberately empty to rule out a deploy explanation. A restart here
would clear nothing, since the problem isn't inside `inventory-service`
at all -- this scenario tests whether the interceptor still gates the
call *even when the action being proposed is a bad idea*, and gives a
human a real reason to reject it, not just rubber-stamp it.

In [6]:
config_2 = {"configurable": {"thread_id": "hitl-inventory-incident"}, "recursion_limit": STEP_BUDGET}
result_2 = gated_investigation_agent.invoke({
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content="inventory-service error rate spiked around 09:14. Investigate and remediate if appropriate."),
    ],
    "report": None,
}, config=config_2)

print("=== TOOL-CALL TRACE ===")
step_index = 0
sensitive_fired_at_2 = None
for m in result_2["messages"]:
    if hasattr(m, "tool_calls") and m.tool_calls:
        for tc in m.tool_calls:
            step_index += 1
            marker = " <-- SENSITIVE" if tc["name"] in SENSITIVE_TOOLS else ""
            print(f"  [{step_index}] called {tc['name']}({tc['args']}){marker}")
            if tc["name"] in SENSITIVE_TOOLS and sensitive_fired_at_2 is None:
                sensitive_fired_at_2 = step_index

if "__interrupt__" in result_2:
    print("\nPAUSED at tool-call index", sensitive_fired_at_2, "-- payload:", result_2["__interrupt__"])
else:
    print("\nNo sensitive tool call requested on this run -- the agent may have correctly recognized a restart wouldn't help.")


=== TOOL-CALL TRACE ===
  [1] called get_service_metrics({'service_name': 'inventory-service', 'metric': 'error_rate'})
  [2] called get_service_logs({'service_name': 'inventory-service'})
  [3] called get_recent_deploys({'service_name': 'inventory-service'})
  [4] called get_dependency_health({'service_name': 'inventory-service'})
  [5] called get_service_metrics({'service_name': 'warehouse-api', 'metric': 'error_rate'})
  [6] called get_service_logs({'service_name': 'warehouse-api'})
  [7] called get_dependency_health({'service_name': 'warehouse-api'})
  [8] called restart_service({'service_name': 'warehouse-api'}) <-- SENSITIVE

PAUSED at tool-call index 8 -- payload: [Interrupt(value={'action': 'restart_service', 'args': {'service_name': 'warehouse-api'}, 'question': "Approve restart_service({'service_name': 'warehouse-api'}) before it executes? This is a real, irreversible action."}, id='327f263044d0d0a649f96e3962e7be88')]


In [7]:
# If the agent proposed a restart here, a human reviewer -- who can see the
# dependency-health evidence right there in the payload -- rejects it: restarting
# inventory-service does nothing about warehouse-api actually being down.
if "__interrupt__" in result_2:
    _human_interventions += 1
    result_2 = gated_investigation_agent.invoke(Command(resume=False), config=config_2)
    print("Resumed after REJECTION -- a restart would not have fixed a downstream dependency failure.")
else:
    print("Nothing to resume -- no sensitive call was ever requested on this run.")

print("\n=== FINAL REPORT ===")
print(result_2["report"])
print("\nRestarts actually executed across BOTH runs:", _RESTARTS_EXECUTED)

scorecard_call_count = scorecard.call_count
scorecard_elapsed_s = _time.time() - _scorecard_start
print(f"\n[Scorecard capture] {scorecard_call_count} real LLM calls, {scorecard_elapsed_s:.2f}s wall-clock across both HITL scenarios")


Resumed after REJECTION -- a restart would not have fixed a downstream dependency failure.

=== FINAL REPORT ===
{'root_cause': 'The `inventory-service` experienced a spike in error rates due to upstream dependency `warehouse-api` returning 503 errors, resulting in a circuit breaker being triggered.', 'affected_service': 'inventory-service', 'contributing_deploy': 'none', 'confidence': 'high', 'recommended_action': 'Notify the team responsible for `warehouse-api` to investigate and resolve the 503 errors.'}

Restarts actually executed across BOTH runs: []

[Scorecard capture] 11 real LLM calls, 16.63s wall-clock across both HITL scenarios


## Standardized single-agent scorecard

The same operational-metrics vocabulary as every other notebook in this
series (`shared.print_scorecard`), captured from this notebook's own real
run above (both HITL scenarios), not recomputed separately.

In [8]:
_total_tool_calls = sum(
    1 for r in [result_1, result_2] for m in r["messages"] if hasattr(m, "tool_calls") and m.tool_calls for _ in m.tool_calls
)
_model_name = OPENAI_MODEL if PROVIDER == "openai" else ANTHROPIC_MODEL
_cost_estimate = shared.estimate_cost_usd(scorecard_call_count, _model_name)

shared.print_scorecard([
    ("Outcome correctness", f"{len(_RESTARTS_EXECUTED)}/1 legitimate restart executed", "Only checkout-service (run 1, warranted) should appear in _RESTARTS_EXECUTED -- inventory-service (run 2, would not have helped) should not"),
    ("Model-call count", str(scorecard_call_count), "Real LLM calls across both scenarios (investigation turns + finalize report)"),
    ("Latency", f"{scorecard_elapsed_s:.2f}s", "Wall-clock for both scenarios, including the pause for human review"),
    ("Estimated cost", f"${_cost_estimate:.4f}", f"shared.estimate_cost_usd({scorecard_call_count} calls, {_model_name}) -- illustrative, not an exact invoice"),
    ("Replans/retries", "N/A", "No replan step -- this is a ReAct loop (see 01_react_single_agent.ipynb), not a planner-executor"),
    ("Tool calls", str(_total_tool_calls), "Total tool calls across both scenarios, including the gated restart_service proposals"),
    ("Human interventions", str(_human_interventions), "Real approve/reject decisions required -- one per scenario where restart_service was proposed"),
    ("Boundary violations", "0 (structural)", "tools_node_with_gate can never execute a SENSITIVE_TOOLS call without interrupt()+Command(resume=True) -- _RESTARTS_EXECUTED is the direct, verifiable proof"),
    ("Context-size proxy", f"{len(ALL_TOOLS)} tools/call (1 sensitive)", "Same tool set as notebook 01 plus restart_service; the gate adds no extra context, only interception logic"),
])


Metric                  Value                 Why it matters
------------------------------------------------------------------------------------------------
Outcome correctness     0/1 legitimate restart executedOnly checkout-service (run 1, warranted) should appear in _RESTARTS_EXECUTED -- inventory-service (run 2, would not have helped) should not
Model-call count        11                    Real LLM calls across both scenarios (investigation turns + finalize report)
Latency                 16.63s                Wall-clock for both scenarios, including the pause for human review
Estimated cost          $0.0020               shared.estimate_cost_usd(11 calls, gpt-4o-mini) -- illustrative, not an exact invoice
Replans/retries         N/A                   No replan step -- this is a ReAct loop (see 01_react_single_agent.ipynb), not a planner-executor
Tool calls              13                    Total tool calls across both scenarios, including the gated restart_service proposals
Hum

**Expected output, and a real, sharper result than an earlier run of this
same notebook produced**: read the actual trace above rather than
assuming a fixed outcome. On this run the agent did not propose restarting
`inventory-service` itself -- it proposed
`restart_service({'service_name': 'warehouse-api'})`, a downstream
dependency this incident's tools have no remediation authority over at
all (`warehouse-api` never appears as a target in any tool this agent was
given other than `restart_service` itself). The interceptor gated it
exactly the same way it would have gated any other `restart_service`
call -- by name, regardless of which service was targeted or whether the
target made any sense -- and a human reviewer rejected it, using the
dependency-health evidence already visible in the interrupt payload.
`_RESTARTS_EXECUTED` stays empty after this run.

Read this precisely: this is an *even stronger* case for the gate's value
than a same-service-wrong-remediation proposal would have been. The
interceptor's check is purely on the tool's name being in
`SENSITIVE_TOOLS` -- it does not, and structurally cannot, reason about
whether the proposed *target* makes sense. A production system relying on
this pattern should treat that as a real, named limitation: the gate
catches "this tool is being called," not "this tool call targets a
sensible service" -- that second judgment is exactly what the human
reviewer's own reasoning has to supply, using the evidence the interrupt
payload surfaces, not something the interceptor verifies on its own.

### Common errors

- **Reusing `ToolNode` and trying to bolt `interrupt()` on around it.**
  `ToolNode` executes every requested call in one internal step with no
  per-call hook -- the interceptor has to replace it, not wrap it.
- **Checking `SENSITIVE_TOOLS` against the wrong thing.** The check has
  to be the tool's *name*, resolved from the actual `tool_calls` the
  model produced -- not, say, a keyword search over the model's free-text
  reasoning, which the model could phrase in a way that doesn't obviously
  mention "restart."
- **Forgetting a checkpointer.** Exactly as in `agent_hitl.ipynb`:
  without one, `interrupt()` has no paused state to resume from -- this
  matters even more here, since the agent's own message history (the
  entire investigation so far) has to survive the pause, not just one
  small state dict.
- **Gating every tool call instead of just the sensitive ones.** Pausing
  before `get_service_logs` too would defeat the whole point of a true
  ReAct agent's free exploration -- the gate belongs on the *specific*
  tools that need it, not on the tool-execution step as a whole.
- **Assuming a gate on tool *identity* also validates tool *target*.**
  This run's real result is direct evidence against that assumption: the
  interceptor correctly paused a `restart_service` call regardless of
  which service it targeted, including a nonsensical target -- but it
  took a human noticing the target was wrong, not the gate itself, to
  catch that specific problem.
- **Assuming the model's free-text recommendation and its actual tool
  calls always agree.** Run 1 (above) is direct evidence they can
  diverge: a report can recommend an action the agent never actually
  requested, which a name-based tool-call interceptor has no way to
  catch, because there was no call to intercept.

## Other design considerations

- **This generalizes past one sensitive tool.** `SENSITIVE_TOOLS` is a
  set precisely so a second mutating tool (e.g. `rollback_deploy`) can be
  added later with a one-line change, still gated by the same
  interceptor, still with no change to the graph's shape.
- **Different sensitive tools could warrant different review payloads.**
  This notebook surfaces `{action, args, question}` uniformly; a real
  system might want `restart_service` to show recent error-rate trend
  data in the same payload, so a reviewer isn't just approving a bare
  function call.
- **This composes with everything notebook 01 already built.** The
  checkpointer used here is the same `InMemorySaver` pattern from that
  notebook's memory section -- a real deployment would layer Mem0/durable
  checkpointing on top exactly as shown there, unchanged by anything in
  this notebook.
- **A rejected action should still be visible in the final report.** This
  notebook's `finalize_node` sees the `ToolMessage` recording the
  rejection ("REJECTED by human reviewer") in the message history, so a
  rejected remediation attempt is not silently dropped from the record --
  it shows up as context the model can (and typically does) mention in
  `recommended_action`.

## Revision summary

- A true ReAct agent has one shared tool-execution step reused on every
  iteration -- there is no fixed graph position to attach a static
  interrupt to for a risky action, because the risky action can be
  requested at any iteration or not at all.
- The fix is a **name-based interceptor inside that one node**: any tool
  call whose name is in `SENSITIVE_TOOLS` triggers `interrupt()`
  regardless of which iteration it's on; everything else executes exactly
  as the prebuilt `ToolNode` would have.
- This required replacing `ToolNode` with a custom function -- the
  prebuilt component has no per-call hook, which is a real, concrete
  limitation worth knowing before reaching for it on a mixed
  read-only/mutating tool set.
- The two real runs in this notebook demonstrate the gate firing at
  different points (or not at all) depending purely on what the agent's
  own investigation concluded -- direct evidence this is a property of
  the tool, not a step count.
- A human rejecting a technically-permitted-but-wrong remediation (run 2)
  is exactly the case a position-based gate can't distinguish from a
  good one -- this interceptor pauses on the tool name every time,
  independent of whether the specific call turns out to be a good idea.

## Checkpoint questions

1. **Q: Why can't notebook 01's `ToolNode` be reused unchanged once
   `restart_service` is added to the tool set?**
   A: `ToolNode` executes every tool call in the agent's message in one
   opaque internal step, with no hook to intervene on a specific call --
   gating requires a custom node that inspects each call individually.

2. **Q: Why is a static `interrupt_before=["tools"]` insufficient here,
   even though it's simpler code than a custom node?**
   A: It would pause before *every* tool call, including harmless
   read-only ones, defeating the free-form ReAct exploration the whole
   agent is built around -- the gate needs to be tool-specific, not
   position-specific.

3. **Q: What determines *when*, if ever, `restart_service` gets flagged
   in a given run?**
   A: Purely the model's own investigation -- there is no fixed
   tool-call index tied to it in the graph; it fires whenever (or if
   ever) the agent's reasoning leads it to request that specific tool.

4. **Q: In run 2, why is it still meaningful for the interceptor to pause
   even if the agent's proposed restart wouldn't actually fix the
   incident?**
   A: Because the gate's job is to make *any* call to a sensitive tool
   reviewable, not to pre-judge whether the call is a good idea -- a
   human with the dependency-health evidence in front of them is exactly
   who should be making that call, not the interceptor itself.

5. **Q: What would happen if `SENSITIVE_TOOLS` checked the model's
   free-text reasoning instead of the actual tool call's name?**
   A: It would be unreliable -- a model could request `restart_service`
   without ever using the word "restart" in its visible reasoning, or
   could mention "restart" in passing without calling the tool; checking
   the structured tool call itself is the only reliable signal.

6. **Q: Why does `finalize_node` still see a rejected remediation attempt
   in the message history?**
   A: Because the rejection is recorded as a real `ToolMessage`
   ("REJECTED by human reviewer") rather than silently discarded -- the
   investigation's message history, which `finalize_node` reads in full,
   includes it like any other tool result.

7. **Q: How would you add a second mutating tool, e.g. `rollback_deploy`,
   to this notebook's design?**
   A: Add it to `ALL_TOOLS` and to the `SENSITIVE_TOOLS` set -- the
   interceptor and the graph shape need no other changes, since the gate
   already checks by name against that set for every call.

8. **Q: Why does this notebook require a checkpointer even more
   critically than `agent_hitl.ipynb`'s small fixed-graph examples did?**
   A: Because the paused state here includes the *entire investigation's*
   message history up to that point, not a small, purpose-built state
   dict -- losing that on resume would mean losing everything the agent
   had already discovered.

9. **Q: What's the concrete difference between this notebook's interceptor
   and `agent_hitl.ipynb`'s conditional interrupt (Part 4 of that
   notebook)?**
   A: `agent_hitl.ipynb`'s conditional interrupt still lived at one known
   node in a fixed graph, conditioned on a threshold; this notebook's
   interceptor lives inside the one node a true ReAct agent reuses for
   *every* tool call, conditioned on which tool was requested -- the
   condition here is about identity, not just risk magnitude, and there
   is no fixed node position it's attached to.

10. **Q: If you removed `restart_service` from `SENSITIVE_TOOLS` but left
    it in `ALL_TOOLS`, what would happen the next time the agent decided
    to call it?**
    A: It would execute immediately, exactly like any read-only tool --
    the interceptor only inspects the set, not anything about the tool's
    actual behavior, so removing it from `SENSITIVE_TOOLS` silently
    removes the gate even though the tool itself is still just as
    irreversible.